모형 1: 항만구분

모형 2: 모형 1 + 총톤수 + 총작업량

모형 3: 모형 2 + 특보유무

모형 4: 모형 3 + 기상변수 6개(결측치 삭제 변수와 동일)

# 1. 자료 불러오기와 변수 설정

In [4]:
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import t
from IPython.display import display

data_dir = Path(r"C:\WORK\stay_multiregression\data")

# 결측 삭제 전 선박자료
ship = pd.read_csv(
    data_dir / "체류시간_찐최종.csv",
    encoding="utf-8-sig",
    low_memory=False,
)

# 원본의 숫자 문자열 정리
numeric_cols = [
    "체류시간",
    "총톤수",
    "총작업량",
    "체류시간_특보노출시간(h)",
]

for col in numeric_cols:
    ship[col] = pd.to_numeric(
        ship[col].astype("string").str.replace(",", "", regex=False),
        errors="raise",
    )

for col in ["접안일시", "출항일시"]:
    ship[col] = pd.to_datetime(ship[col], errors="raise")

ship["항만구분"] = ship["항만구분"].str.strip()

ship["신항더미"] = ship["항만구분"].map({
    "북항": 0,
    "신항": 1,
})

# 이번 기상 MICE에서 대치하지 않은 기본정보 확인
check_cols = numeric_cols + [
    "접안일시", "출항일시", "신항더미"
]

if ship[check_cols].isna().any().any():
    raise ValueError("선박 기본정보 또는 특보노출시간에 결측이 있습니다.")

if not np.isfinite(ship[numeric_cols].to_numpy(dtype=float)).all():
    raise ValueError("선박 기본정보에 무한값이 있습니다.")

if (ship["출항일시"] <= ship["접안일시"]).any():
    raise ValueError("접안일시와 출항일시를 확인하세요.")

if (ship["체류시간_특보노출시간(h)"] < 0).any():
    raise ValueError("특보노출시간에 음수가 있습니다.")

ship["특보유무"] = (
    ship["체류시간_특보노출시간(h)"] > 0
).astype(int)

# 결측 삭제 버전과 동일한 기상변수 6개
weather_vars = [
    "체류시간_풍속_중앙값_mps",
    "체류시간_최대순간풍속_최댓값_mps",
    "체류시간_유의파고_중앙값_m",
    "체류시간_순간최대파고_최댓값_m",
    "체류시간_시정_최솟값_10m",
    "체류시간_시정_중앙값_10m",
]

# 결측 삭제 버전과 동일한 4단계 구성
vars_1 = ["신항더미"]
vars_2 = vars_1 + ["총톤수", "총작업량"]
vars_3 = vars_2 + ["특보유무"]
vars_4 = vars_3 + weather_vars

model_vars = {
    1: vars_1,
    2: vars_2,
    3: vars_3,
    4: vars_4,
}

mice_files = sorted(
    (data_dir / "MICE").glob("weather_2021_2024_MICE_*.csv")
)

if len(mice_files) != 50:
    raise ValueError(f"MICE 파일 수를 확인하세요: {len(mice_files)}개")

print(f"선박자료: {len(ship):,}행")
print(f"MICE 기상자료: {len(mice_files)}벌")
print(f"모형 4 독립변수: {len(vars_4)}개")  # 10개

선박자료: 33,994행
MICE 기상자료: 50벌
모형 4 독립변수: 10개


# 2. 기상변수 6개 집계 → 대치자료별 4단계 OLS 적합
집계 구간은 접안일시 ≤ 관측시각 < 출항일시입니다.

In [5]:
END_SIDE = "right"

source_cols = [
    "풍속(m/s)",
    "최대순간풍속(m/s)",
    "파고_유의파고(m)",
    "파고_최대파고(m)",
    "시정(10m)",
]


def aggregate_weather(weather):
    # 기상변수 6개를 저장할 공간
    result = np.full((len(ship), len(weather_vars)), np.nan)

    for port, rows in ship.groupby("항만구분"):
        w = (
            weather.loc[weather["항만"] == port]
            .sort_values("관측시각")
        )

        if w.empty:
            raise ValueError(f"{port} 기상자료가 없습니다.")

        times = w["관측시각"].to_numpy(dtype="datetime64[ns]")
        values = w[source_cols].to_numpy(dtype=float)

        if not np.isfinite(values).all():
            raise ValueError(f"{port} 기상자료에 결측·무한값이 있습니다.")

        if not np.all(np.diff(times) == np.timedelta64(1, "h")):
            raise ValueError(f"{port} 관측시각에 중복 또는 공백이 있습니다.")

        starts = rows["접안일시"].to_numpy(dtype="datetime64[ns]")
        ends = rows["출항일시"].to_numpy(dtype="datetime64[ns]")

        if (
            (starts < times[0]).any()
            or (ends > times[-1] + np.timedelta64(1, "h")).any()
        ):
            raise ValueError(f"{port} 체류기간이 기상자료 범위를 벗어납니다.")

        lo = np.searchsorted(times, starts, side="left")
        hi = np.searchsorted(times, ends, side=END_SIDE)

        if (hi <= lo).any():
            raise ValueError(f"{port}에 기상 관측이 없는 체류기간이 있습니다.")

        positions = lo[:, None] + np.arange((hi - lo).max())
        inside = positions < hi[:, None]

        block = values[np.minimum(positions, len(values) - 1)].copy()
        block[~inside] = np.nan

        # 결측 삭제 모형과 같은 6개 변수만 계산
        result[rows.index] = np.column_stack([
            np.nanmedian(block[:, :, 0], axis=1),  # 풍속 중앙값
            np.nanmax(block[:, :, 1], axis=1),     # 최대순간풍속 최댓값
            np.nanmedian(block[:, :, 2], axis=1),  # 유의파고 중앙값
            np.nanmax(block[:, :, 3], axis=1),     # 최대파고 최댓값
            np.nanmin(block[:, :, 4], axis=1),     # 시정 최솟값
            np.nanmedian(block[:, :, 4], axis=1),  # 시정 중앙값
        ])

    if not np.isfinite(result).all():
        raise ValueError("기상 집계 후 결측값이 남았습니다.")

    return result


# 재실행할 때 이전 9개 변수 분석 결과가 섞이지 않도록 초기화
estimates = {number: [] for number in model_vars}
covariances = {number: [] for number in model_vars}
complete_df = {}
parameter_names = {}
comparison_records = []

for number, file in enumerate(mice_files, start=1):
    weather = pd.read_csv(file, encoding="utf-8-sig")
    weather["관측시각"] = pd.to_datetime(
        weather["관측시각"], errors="raise"
    )
    weather["항만"] = weather["항만"].str.strip()

    data = ship[["체류시간"] + vars_3].astype(float).copy()
    data[weather_vars] = aggregate_weather(weather)

    previous_r2 = 0

    for model_number, columns in model_vars.items():
        X = sm.add_constant(data[columns], has_constant="add")
        fit = sm.OLS(data["체류시간"], X).fit()

        estimates[model_number].append(fit.params.to_numpy())
        covariances[model_number].append(fit.cov_params().to_numpy())

        complete_df[model_number] = fit.df_resid
        parameter_names[model_number] = fit.params.index

        comparison_records.append({
            "대치번호": number,
            "모형": model_number,
            "표본 수": int(fit.nobs),
            "독립변수 수": len(columns),
            "R²": fit.rsquared,
            "수정 R²": fit.rsquared_adj,
            "R² 증가": (
                0 if model_number == 1
                else fit.rsquared - previous_r2
            ),
        })

        previous_r2 = fit.rsquared

    print(f"{number}/50 완료")

1/50 완료
2/50 완료
3/50 완료
4/50 완료
5/50 완료
6/50 완료
7/50 완료
8/50 완료
9/50 완료
10/50 완료
11/50 완료
12/50 완료
13/50 완료
14/50 완료
15/50 완료
16/50 완료
17/50 완료
18/50 완료
19/50 완료
20/50 완료
21/50 완료
22/50 완료
23/50 완료
24/50 완료
25/50 완료
26/50 완료
27/50 완료
28/50 완료
29/50 완료
30/50 완료
31/50 완료
32/50 완료
33/50 완료
34/50 완료
35/50 완료
36/50 완료
37/50 완료
38/50 완료
39/50 완료
40/50 완료
41/50 완료
42/50 완료
43/50 완료
44/50 완료
45/50 완료
46/50 완료
47/50 완료
48/50 완료
49/50 완료
50/50 완료


# 3. 모형별 Rubin 통합 결과와 설명력 요약

In [6]:
def pool_results(model_number):
    q = np.stack(estimates[model_number])
    u = np.mean(covariances[model_number], axis=0)
    m = len(q)

    coef = q.mean(axis=0)
    between = np.cov(q, rowvar=False, ddof=1)
    total = u + (1 + 1 / m) * between
    se = np.sqrt(np.diag(total))

    # Barnard-Rubin 자유도 보정
    lam = (1 + 1 / m) * np.diag(between) / np.diag(total)
    lam = np.clip(lam, 0, 1 - 1e-12)

    df_old = np.full_like(lam, np.inf)
    np.divide(m - 1, lam ** 2, out=df_old, where=lam > 0)

    nu = complete_df[model_number]
    df_observed = ((nu + 1) / (nu + 3)) * nu * (1 - lam)
    df_adjusted = 1 / (1 / df_old + 1 / df_observed)

    critical = t.ppf(0.975, df_adjusted)
    pvalues = 2 * t.sf(np.abs(coef / se), df_adjusted)

    return pd.DataFrame({
        "통합 회귀계수": coef,
        "통합 표준오차": se,
        "95% CI 하한": coef - critical * se,
        "95% CI 상한": coef + critical * se,
        "p값": pvalues,
    }, index=parameter_names[model_number]).rename(
        index={"const": "절편"}
    )


pooled_tables = {}

for number in model_vars:
    pooled_tables[number] = pool_results(number)

    print(f"\n모형 {number}: 다중대치 통합 결과")
    display(
        pooled_tables[number].style.format({
            "통합 회귀계수": "{:.6g}",
            "통합 표준오차": "{:.6g}",
            "95% CI 하한": "{:.6g}",
            "95% CI 상한": "{:.6g}",
            "p값": lambda p: "<0.001" if p < 0.001 else f"{p:.3f}",
        })
    )

comparison_detail = pd.DataFrame(comparison_records)

comparison = (
    comparison_detail.groupby("모형")
    .agg(
        표본수=("표본 수", "first"),
        독립변수수=("독립변수 수", "first"),
        평균_R2=("R²", "mean"),
        최소_R2=("R²", "min"),
        최대_R2=("R²", "max"),
        평균_수정R2=("수정 R²", "mean"),
        평균_R2증가=("R² 증가", "mean"),
    )
)

print("\n50개 대치자료의 설명력 요약")
display(
    comparison.style.format({
        "표본수": "{:,.0f}",
        "독립변수수": "{:.0f}",
        "평균_R2": "{:.4f}",
        "최소_R2": "{:.4f}",
        "최대_R2": "{:.4f}",
        "평균_수정R2": "{:.4f}",
        "평균_R2증가": "{:.4f}",
    })
)


모형 1: 다중대치 통합 결과


,통합 회귀계수,통합 표준오차,95% CI 하한,95% CI 상한,p값
절편,17.6721,0.0692961,17.5363,17.8079,<0.001
신항더미,4.77899,0.0996731,4.58362,4.97435,<0.001



모형 2: 다중대치 통합 결과


,통합 회귀계수,통합 표준오차,95% CI 하한,95% CI 상한,p값
절편,12.7641,0.0475271,12.671,12.8573,<0.001
신항더미,-4.26493,0.0802299,-4.42218,-4.10768,<0.001
총톤수,-2.0844e-05,1.17758e-06,-2.31521e-05,-1.85359e-05,<0.001
총작업량,0.00920908,5.09139e-05,0.00910929,0.00930888,<0.001



모형 3: 다중대치 통합 결과


,통합 회귀계수,통합 표준오차,95% CI 하한,95% CI 상한,p값
절편,12.5604,0.0482825,12.4658,12.655,<0.001
신항더미,-4.2485,0.0797479,-4.40481,-4.0922,<0.001
총톤수,-2.05822e-05,1.17052e-06,-2.28764e-05,-1.82879e-05,<0.001
총작업량,0.00914829,5.06931e-05,0.00904893,0.00924765,<0.001
특보유무,1.78888,0.0876462,1.61709,1.96067,<0.001



모형 4: 다중대치 통합 결과


,통합 회귀계수,통합 표준오차,95% CI 하한,95% CI 상한,p값
절편,9.92305,0.130563,9.6671,10.179,<0.001
신항더미,-3.42037,0.0814059,-3.57994,-3.26081,<0.001
총톤수,-1.84262e-05,1.09997e-06,-2.05822e-05,-1.62703e-05,<0.001
총작업량,0.00815057,5.08921e-05,0.00805081,0.00825032,<0.001
특보유무,0.697393,0.108639,0.484414,0.910373,<0.001
체류시간_풍속_중앙값_mps,-1.29702,0.0466643,-1.38853,-1.20551,<0.001
체류시간_최대순간풍속_최댓값_mps,0.932668,0.0252055,0.883194,0.982142,<0.001
체류시간_유의파고_중앙값_m,-4.66391,0.303484,-5.2657,-4.06213,<0.001
체류시간_순간최대파고_최댓값_m,2.13105,0.16215,1.80819,2.4539,<0.001
체류시간_시정_최솟값_10m,-0.000897805,2.86322e-05,-0.000953932,-0.000841679,<0.001



50개 대치자료의 설명력 요약


,표본수,독립변수수,평균_R2,최소_R2,최대_R2,평균_수정R2,평균_R2증가
모형,,,,,,,
1,"33,994",1,0.0633,0.0633,0.0633,0.0633,0.0000
2,"33,994",3,0.6470,0.6470,0.6470,0.6470,0.5836
3,"33,994",4,0.6513,0.6513,0.6513,0.6512,0.0043
4,"33,994",10,0.6933,0.6913,0.6946,0.6932,0.0420


# 결측치 삭제 버전 vs MICE 버전
 회귀계수, p값, R²를 한 표에서 비교

R²는 변수별 수치가 아니라 모형 전체의 수치이므로 같은 모형의 행마다 동일하게 표시됩니다. MICE 쪽은 50벌의 평균 R²이며, 회귀계수·p값은 Rubin 통합 결과입니다.

In [7]:
import pandas as pd
import statsmodels.api as sm
from IPython.display import display

# 1. 결측 삭제 버전 재현
# 기존 방식과 동일하게 원본의 모든 열을 기준으로 결측 행 삭제
cc_data = pd.read_csv(
    data_dir / "체류시간_찐최종.csv",
    encoding="utf-8-sig",
    low_memory=False,
).dropna().copy()

# 항만 더미
cc_data["신항더미"] = (
    cc_data["항만구분"].str.strip()
    .map({"북항": 0, "신항": 1})
)

# 분석에 필요한 숫자 열 정리
numeric_cols = [
    "체류시간",
    "총톤수",
    "총작업량",
    "체류시간_특보노출시간(h)",
] + weather_vars

for col in numeric_cols:
    cc_data[col] = pd.to_numeric(
        cc_data[col].astype("string").str.replace(",", "", regex=False),
        errors="raise",
    )

# 특보유무
cc_data["특보유무"] = (
    cc_data["체류시간_특보노출시간(h)"] > 0
).astype(int)

if cc_data[["체류시간"] + vars_4].isna().any().any():
    raise ValueError("결측 삭제 자료의 분석변수에 결측값이 있습니다.")

# 2. MICE와 동일한 변수 구성으로 결측 삭제 모형 적합
cc_models = {}

for number, columns in model_vars.items():
    X_cc = sm.add_constant(
        cc_data[columns].astype(float),
        has_constant="add",
    )

    cc_models[number] = sm.OLS(
        cc_data["체류시간"],
        X_cc,
    ).fit()

# 3. MICE 자료 50벌의 평균 R²
mice_r2 = comparison_detail.groupby("모형")["R²"].mean()

# 4. 모형별·변수별 결과를 한 표로 합치기
records = []

for number, cc_model in cc_models.items():
    cc_coef = cc_model.params.rename(index={"const": "절편"})
    cc_pvalue = cc_model.pvalues.rename(index={"const": "절편"})

    mice_result = pooled_tables[number]

    # 두 분석의 변수가 같은지 확인
    if set(cc_coef.index) != set(mice_result.index):
        raise ValueError(f"모형 {number}: 두 분석의 변수 구성이 다릅니다.")

    for variable in cc_coef.index:
        records.append({
            "모형": f"모형 {number}",
            "변수": variable,
            "삭제 회귀계수": cc_coef[variable],
            "삭제 p값": cc_pvalue[variable],
            "삭제 R²": cc_model.rsquared,
            "MICE 통합 회귀계수": mice_result.loc[
                variable, "통합 회귀계수"
            ],
            "MICE 통합 p값": mice_result.loc[variable, "p값"],
            "MICE 평균 R²": mice_r2[number],
        })

result_comparison = (
    pd.DataFrame(records)
    .set_index(["모형", "변수"])
)

# 5. 비교표 출력
print(f"결측 삭제 표본 수: {len(cc_data):,}")
print(
    "MICE 표본 수:",
    f"{int(comparison_detail['표본 수'].iloc[0]):,}",
)

display(
    result_comparison.style
    .format({
        "삭제 회귀계수": "{:.6g}",
        "삭제 p값": lambda p: "<0.001" if p < 0.001 else f"{p:.3f}",
        "삭제 R²": "{:.4f}",
        "MICE 통합 회귀계수": "{:.6g}",
        "MICE 통합 p값": lambda p: "<0.001" if p < 0.001 else f"{p:.3f}",
        "MICE 평균 R²": "{:.4f}",
    })
    .set_properties(**{"padding": "8px 10px"})
)

결측 삭제 표본 수: 32,014
MICE 표본 수: 33,994


회귀계수·표준오차·신뢰구간·p값은 통합 결과이고, 마지막 표의 평균 R²는 50개 분석의 기술적 요약입니다. 평균 R²를 Rubin 방식으로 통합한 검정 결과라고 해석하지는 않습니다.

또한 이 코드는 제공된 대치자료를 분석하는 절차입니다. 대치모형이 최종 회귀관계를 충분히 보존했는지와 시간적 의존성은 별도로 검토해야 합니다.

이제 결측치 삭제 버전이랑 MICE 버전의 회귀계수, p값, R²를 한 표에서 비교하는 코드 만들어줘

# 추가 검증 (삭제, MICE 중 뭐가 더 좋을까요?)

1. 검증용 자료 생성
2021년 1~2월의 실제 결측 패턴을 참고하여, 2022·2023·2024년의 같은 기간에서 관측값을 가립니다. 원본 파일은 변경하지 않습니다.

In [8]:
from pathlib import Path
import numpy as np
import pandas as pd

# 파일 위치
root = Path(r"C:\WORK\stay_multiregression")
raw_path = Path(r"C:\Users\user\Downloads\weather_final.csv")
validation_dir = root / "data" / "MICE_검증"

# 결측값이 남아 있는 시간별 원본 기상자료 불러오기
for encoding in ["utf-8-sig", "cp949", "euc-kr"]:
    try:
        raw = pd.read_csv(
            raw_path,
            encoding=encoding,
            low_memory=False,
        )
        break
    except UnicodeDecodeError:
        continue
else:
    raise ValueError("원본 기상 CSV의 인코딩을 확인하세요.")

keys = ["항만", "관측시각"]

weather_cols = [
    "최대순간풍속(m/s)",
    "풍속(m/s)",
    "파고_유의파고(m)",
    "파고_최대파고(m)",
    "시정(10m)",
    "풍향(8방위)",
]

numeric_cols = weather_cols[:-1]

raw["관측시각"] = pd.to_datetime(
    raw["관측시각"], errors="raise"
)
raw["항만"] = raw["항만"].str.strip()

# 2021~2024년 자료만 사용
raw = raw[
    raw["관측시각"].dt.year.between(2021, 2024)
].copy()

# 숫자형 기상변수 정리
for col in numeric_cols:
    raw[col] = pd.to_numeric(
        raw[col]
        .astype("string")
        .str.replace(",", "", regex=False),
        errors="raise",
    )

raw[numeric_cols] = raw[numeric_cols].replace(
    [np.inf, -np.inf], np.nan
)

if raw.duplicated(keys).any():
    raise ValueError("동일한 항만·관측시각의 기록이 중복되어 있습니다.")

# 시간별 관측 행 자체가 없는 경우도 결측으로 표시
grid = pd.MultiIndex.from_product(
    [
        ["북항", "신항"],
        pd.date_range(
            "2021-01-01 00:00:00",
            "2024-12-31 23:00:00",
            freq="h",
        ),
    ],
    names=keys,
)

raw = (
    raw.set_index(keys)[weather_cols]
    .reindex(grid)
    .reset_index()
)

# 평가에 사용할 원래 관측값 보관
validation_dir.mkdir(parents=True, exist_ok=True)

raw.to_csv(
    validation_dir / "원래_관측자료.csv",
    index=False,
    encoding="utf-8-sig",
)

# 2021년 1~2월의 실제 결측 위치 확인
template = raw[
    (raw["관측시각"].dt.year == 2021)
    & (raw["관측시각"].dt.month <= 2)
].copy()

template["월일시각"] = (
    template["관측시각"].dt.strftime("%m-%d %H:%M")
)

template_missing = template.set_index(
    ["항만", "월일시각"]
)[weather_cols].isna()

# 다른 해에 같은 결측 패턴을 적용
for year in [2022, 2023, 2024]:
    folder = validation_dir / str(year)
    folder.mkdir(parents=True, exist_ok=True)

    masked = raw.copy()

    test_period = (
        (raw["관측시각"].dt.year == year)
        & (raw["관측시각"].dt.month <= 2)
    )

    target = raw.loc[test_period, keys].copy()

    target["월일시각"] = (
        target["관측시각"].dt.strftime("%m-%d %H:%M")
    )

    # 2024년 2월 29일은 2월 28일의 결측 패턴 적용
    target["월일시각"] = target["월일시각"].str.replace(
        "02-29", "02-28", regex=False
    )

    matched_pattern = template_missing.reindex(
        pd.MultiIndex.from_frame(
            target[["항만", "월일시각"]]
        )
    )

    if matched_pattern.isna().any().any():
        raise ValueError("2021년 결측 패턴과 연결할 수 없는 시각이 있습니다.")

    # 실제값을 가린 위치만 True로 저장
    score_mask = pd.DataFrame(
        False,
        index=raw.index,
        columns=weather_cols,
    )

    for col in weather_cols:
        # 원래 관측값이 있는 위치만 가림
        hide = (
            matched_pattern[col].to_numpy(dtype=bool)
            & raw.loc[test_period, col].notna().to_numpy()
        )

        hidden_rows = raw.index[test_period][hide]

        masked.loc[hidden_rows, col] = np.nan
        score_mask.loc[hidden_rows, col] = True

    # MICE 프로그램에 입력할 검증자료
    masked.to_csv(
        folder / "검증입력.csv",
        index=False,
        encoding="utf-8-sig",
    )

    # 나중에 실제값과 비교할 위치
    pd.concat([raw[keys], score_mask], axis=1).to_csv(
        folder / "평가위치.csv",
        index=False,
        encoding="utf-8-sig",
    )

    (folder / "MICE결과").mkdir(exist_ok=True)

    print(f"\n{year}년 검증자료 생성 완료")
    print("인위적으로 가린 관측값 수:")
    print(score_mask.sum())


2022년 검증자료 생성 완료
인위적으로 가린 관측값 수:
최대순간풍속(m/s)    2832
풍속(m/s)          26
파고_유의파고(m)       66
파고_최대파고(m)       66
시정(10m)           0
풍향(8방위)          26
dtype: int64

2023년 검증자료 생성 완료
인위적으로 가린 관측값 수:
최대순간풍속(m/s)    2825
풍속(m/s)          33
파고_유의파고(m)       63
파고_최대파고(m)       63
시정(10m)           0
풍향(8방위)          33
dtype: int64

2024년 검증자료 생성 완료
인위적으로 가린 관측값 수:
최대순간풍속(m/s)    2880
풍속(m/s)          36
파고_유의파고(m)       67
파고_최대파고(m)       67
시정(10m)           0
풍향(8방위)          36
dtype: int64


2. 기존 MICE 프로그램으로 다시 대치

In [10]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from scipy.stats import invwishart
from sklearn.linear_model import LogisticRegression

NUM = ['최대순간풍속(m/s)', '풍속(m/s)', '파고_유의파고(m)',
       '파고_최대파고(m)', '시정(10m)']
DIR = '풍향(8방위)'
LEVELS = ['북', '북동', '동', '남동', '남', '남서', '서', '북서']


class PortMICE:
    def __init__(self, frame, port, seed, donors=5):
        self.frame = frame.reset_index(drop=True).copy()
        self.port = port
        self.rng = np.random.default_rng(seed)
        self.donors = donors
        self.original = self.frame[NUM].to_numpy(dtype=float)
        if np.isinf(self.original).any() or (self.original < 0).any():
            raise ValueError(f'{port}: 기상값에 무한값 또는 음수가 있습니다.')
        self.missing = np.isnan(self.original)
        self.current = self.original.copy()
        for j in range(5):
            observed = self.original[~self.missing[:, j], j]
            if len(observed) < 30:
                raise ValueError(f'{port}: {NUM[j]}의 관측값이 부족합니다.')
            self.current[self.missing[:, j], j] = self.rng.choice(
                observed, self.missing[:, j].sum())

        direction = self.frame[DIR]
        bad = direction.notna() & ~direction.isin(LEVELS)
        if bad.any():
            raise ValueError(f'알 수 없는 풍향: {direction[bad].unique()}')
        self.direction_missing = direction.isna().to_numpy()
        codes = direction.map({s: i for i, s in enumerate(LEVELS)})
        self.direction = codes.fillna(-1).to_numpy(dtype=int)
        self.direction_original = self.direction.copy()
        observed = self.direction[~self.direction_missing]
        if not len(observed):
            raise ValueError(f'{port}: 관측된 풍향이 없습니다.')
        self.direction[self.direction_missing] = self.rng.choice(
            observed, self.direction_missing.sum())

        stamp = pd.to_datetime(self.frame['관측시각'])
        year_length = np.where(stamp.dt.is_leap_year, 366, 365)
        season = 2*np.pi*(stamp.dt.dayofyear.to_numpy()-1)/year_length
        hour = stamp.dt.hour.to_numpy()+stamp.dt.minute.to_numpy()/60
        clock = 2*np.pi*hour/24
        self.time = np.column_stack([
            np.sin(season), np.cos(season), np.sin(clock), np.cos(clock),
            stamp.dt.year.to_numpy()-2021])

    def design(self, targets=(), for_direction=False):
        keep = [j for j in range(5) if j not in targets]
        parts = [np.log1p(self.current[:, keep]), self.time]
        if not for_direction:
            parts.append(np.eye(8)[self.direction][:, 1:])
        z = np.column_stack(parts)
        sd = z.std(axis=0)
        z = z[:, sd > 1e-12]
        z = (z-z.mean(axis=0))/z.std(axis=0)
        return np.column_stack([np.ones(len(z)), z])

    def allowed(self, donor_values, row, targets):
        ok = np.ones(len(donor_values), dtype=bool)
        if len(targets) == 2:
            if targets == (2, 3):
                ok &= donor_values[:, 1] >= donor_values[:, 0]
            elif targets == (0, 1) and self.port == '신항':
                ok &= donor_values[:, 0] >= donor_values[:, 1]
        else:
            j = targets[0]
            v = donor_values[:, 0]
            if j == 2:
                ok &= v <= self.current[row, 3]
            elif j == 3:
                ok &= v >= self.current[row, 2]
            elif j == 0 and self.port == '신항':
                ok &= v >= self.current[row, 1]
            elif j == 1 and self.port == '신항':
                ok &= v <= self.current[row, 0]
        return ok

    def pmm(self, targets, recipients):
        rows = np.flatnonzero(recipients)
        if not len(rows):
            return
        observed = ~self.missing[:, list(targets)].any(axis=1)
        donor_rows = np.flatnonzero(observed)
        design = self.design(targets)
        xo = design[observed]
        yo = np.log1p(self.original[np.ix_(donor_rows, targets)])
        n, p = xo.shape
        q = len(targets)
        if n-p <= q+1:
            raise ValueError(f'{self.port}: PMM 관측 표본이 부족합니다.')

        # 약한 ridge 안정화 + 회귀모수의 확률적 추출
        gram = xo.T @ xo
        gram += np.diag(np.maximum(np.diag(gram), 1)*1e-5)
        inverse = np.linalg.inv(gram)
        inverse = (inverse+inverse.T)/2
        beta = inverse @ xo.T @ yo
        error = yo-xo @ beta
        sigma = np.atleast_2d(invwishart.rvs(
            df=n-p, scale=error.T @ error+np.eye(q)*1e-8,
            random_state=self.rng))
        draw = beta + np.linalg.cholesky(inverse) @ self.rng.normal(
            size=(p, q)) @ np.linalg.cholesky(sigma).T

        donor_pred = xo @ beta
        receiver_pred = design[rows] @ draw
        # 블록의 두 예측값은 척도를 맞춘 뒤 유클리드 거리로 매칭
        scale = np.maximum(donor_pred.std(axis=0), 1e-8)
        tree = cKDTree(donor_pred/scale)
        values = self.original[np.ix_(donor_rows, targets)]
        n_donors = len(values)
        k = min(self.donors, n_donors)
        _, initial = tree.query(receiver_pred/scale, k=k)
        initial = np.asarray(initial).reshape(len(rows), k)

        for i, row in enumerate(rows):
            candidates = initial[i]
            size = k
            while True:
                usable = candidates[self.allowed(values[candidates], row, targets)]
                if len(usable) >= k or size == n_donors:
                    break
                size = min(n_donors, max(size*2, k+1))
                _, candidates = tree.query(receiver_pred[i]/scale, k=size)
                candidates = np.atleast_1d(candidates)
            if not len(usable):
                raise ValueError(
                    f'{self.port}, 행 {row}: 물리조건을 만족하는 기증값이 없습니다.')
            selected = self.rng.choice(usable[:k])
            self.current[row, list(targets)] = values[selected]

    def update_direction(self):
        if not self.direction_missing.any():
            return
        obs = np.flatnonzero(~self.direction_missing)
        # 부트스트랩으로 분류모형 추정의 변동도 반영
        sample = self.rng.choice(obs, len(obs), replace=True)
        design = self.design(for_direction=True)[:, 1:]
        if len(np.unique(self.direction_original[sample])) == 1:
            self.direction[self.direction_missing] = self.direction_original[sample[0]]
            return
        model = LogisticRegression(C=1.0, solver='lbfgs', max_iter=500)
        model.fit(design[sample], self.direction_original[sample])
        if np.max(model.n_iter_) >= 500:
            raise RuntimeError('풍향 분류모형이 수렴하지 않았습니다.')
        probability = model.predict_proba(design[self.direction_missing])
        u = self.rng.random((len(probability), 1))
        chosen = (u > probability.cumsum(axis=1)).sum(axis=1)
        self.direction[self.direction_missing] = model.classes_[
            np.minimum(chosen, len(model.classes_)-1)]

    def step(self):
        for pair in [(0, 1), (2, 3)]:
            both = self.missing[:, list(pair)].all(axis=1)
            self.pmm(pair, both)
            for j in pair:
                self.pmm((j,), self.missing[:, j] & ~both)
        self.pmm((4,), self.missing[:, 4])
        self.update_direction()

    def completed(self):
        if not np.isfinite(self.current).all():
            raise ValueError('대치 후 비정상 숫자가 있습니다.')
        np.testing.assert_array_equal(
            self.current[~self.missing], self.original[~self.missing])
        np.testing.assert_array_equal(
            self.direction[~self.direction_missing],
            self.direction_original[~self.direction_missing])
        wave_imputed = self.missing[:, 2] | self.missing[:, 3]
        if (self.current[wave_imputed, 3] < self.current[wave_imputed, 2]).any():
            raise ValueError('대치된 파고의 크기 조건이 위반됐습니다.')
        if self.port == '신항':
            wind_imputed = self.missing[:, 0] | self.missing[:, 1]
            if (self.current[wind_imputed, 0] < self.current[wind_imputed, 1]).any():
                raise ValueError('신항 대치 풍속의 크기 조건이 위반됐습니다.')
        out = self.frame.copy()
        out[NUM] = self.current
        out[DIR] = np.asarray(LEVELS)[self.direction]
        return out


def run_validation(root, years=(2022, 2023, 2024), m=50,
                   burn_in=20, gap=2, seed=20261107):
    root = Path(root)
    for year in years:
        source = root/str(year)/'검증입력.csv'
        output = root/str(year)/'MICE결과'
        output.mkdir(parents=True, exist_ok=True)
        if list(output.glob('*.csv')):
            raise FileExistsError(f'{output}: 기존 CSV를 별도 보관한 뒤 실행하세요.')
        data = pd.read_csv(source, encoding='utf-8-sig')
        data['관측시각'] = pd.to_datetime(data['관측시각'], errors='raise')
        if data.duplicated(['항만', '관측시각']).any():
            raise ValueError('항만·관측시각이 중복됩니다.')
        if set(data['항만'].unique()) != {'신항', '북항'}:
            raise ValueError('항만 이름을 확인하세요.')
        engines = [PortMICE(data.loc[data['항만'].eq(port)].sort_values('관측시각'),
                            port, seed+year*10+i)
                   for i, port in enumerate(['북항', '신항'])]
        trace = []
        saved = 0
        for iteration in range(1, burn_in+gap*m+1):
            for engine in engines:
                engine.step()
                for j, col in enumerate(NUM):
                    v = engine.current[engine.missing[:, j], j]
                    if len(v):
                        trace.append({'반복': iteration, '항만': engine.port,
                                      '변수': col, '평균': v.mean(),
                                      '표준편차': v.std(), '95분위수': np.quantile(v, .95)})
            if iteration > burn_in and (iteration-burn_in) % gap == 0:
                saved += 1
                completed = pd.concat([e.completed() for e in engines], ignore_index=True)
                completed.to_csv(output/f'weather_2021_2024_MICE_{saved:02}.csv',
                                 index=False, encoding='utf-8-sig')
                print(f'{year}년: {saved}/{m}벌 저장 (반복 {iteration})', flush=True)
            elif iteration % 5 == 0:
                print(f'{year}년: 초기 반복 {iteration}/{burn_in}', flush=True)
        # 평가 코드가 결과 폴더의 CSV 50개만 읽도록 진단파일은 상위 폴더에 저장
        pd.DataFrame(trace).to_csv(
            root/str(year)/'반복진단.csv', index=False, encoding='utf-8-sig')
        settings = {'대치수': m, '초기반복': burn_in, '저장간격': gap,
                    '기증후보수': 5, '난수시드': seed,
                    '풍향': '부트스트랩 다항 로지스틱 + 확률 추출',
                    '연속형': 'log1p, 확률적 회귀모수, 단일/블록 PMM',
                    '입력_SHA256': hashlib.sha256(source.read_bytes()).hexdigest()}
        (root/str(year)/'실행설정.json').write_text(
            json.dumps(settings, ensure_ascii=False, indent=2), encoding='utf-8')


if __name__ == '__main__':
    run_validation(r'C:\WORK\stay_multiregression\data\MICE_검증')


C:\Users\user\AppData\Local\Temp\ipykernel_28436\2888005364.py:50: Pandas4Warning: Series.dt.dayofyear is deprecated and will be removed in a future version. Use Series.dt.day_of_year instead.
  season = 2*np.pi*(stamp.dt.dayofyear.to_numpy()-1)/year_length
C:\Users\user\AppData\Local\Temp\ipykernel_28436\2888005364.py:50: Pandas4Warning: Series.dt.dayofyear is deprecated and will be removed in a future version. Use Series.dt.day_of_year instead.
  season = 2*np.pi*(stamp.dt.dayofyear.to_numpy()-1)/year_length


2022년: 초기 반복 5/20
2022년: 초기 반복 10/20
2022년: 초기 반복 15/20
2022년: 초기 반복 20/20
2022년: 1/50벌 저장 (반복 22)
2022년: 2/50벌 저장 (반복 24)
2022년: 초기 반복 25/20
2022년: 3/50벌 저장 (반복 26)
2022년: 4/50벌 저장 (반복 28)
2022년: 5/50벌 저장 (반복 30)
2022년: 6/50벌 저장 (반복 32)
2022년: 7/50벌 저장 (반복 34)
2022년: 초기 반복 35/20
2022년: 8/50벌 저장 (반복 36)
2022년: 9/50벌 저장 (반복 38)
2022년: 10/50벌 저장 (반복 40)
2022년: 11/50벌 저장 (반복 42)
2022년: 12/50벌 저장 (반복 44)
2022년: 초기 반복 45/20
2022년: 13/50벌 저장 (반복 46)
2022년: 14/50벌 저장 (반복 48)
2022년: 15/50벌 저장 (반복 50)
2022년: 16/50벌 저장 (반복 52)
2022년: 17/50벌 저장 (반복 54)
2022년: 초기 반복 55/20
2022년: 18/50벌 저장 (반복 56)
2022년: 19/50벌 저장 (반복 58)
2022년: 20/50벌 저장 (반복 60)
2022년: 21/50벌 저장 (반복 62)
2022년: 22/50벌 저장 (반복 64)
2022년: 초기 반복 65/20
2022년: 23/50벌 저장 (반복 66)
2022년: 24/50벌 저장 (반복 68)
2022년: 25/50벌 저장 (반복 70)
2022년: 26/50벌 저장 (반복 72)
2022년: 27/50벌 저장 (반복 74)
2022년: 초기 반복 75/20
2022년: 28/50벌 저장 (반복 76)
2022년: 29/50벌 저장 (반복 78)
2022년: 30/50벌 저장 (반복 80)
2022년: 31/50벌 저장 (반복 82)
2022년: 32/50벌 저장 (반복 84)
2022년: 초기 반복 85/20
2

C:\Users\user\AppData\Local\Temp\ipykernel_28436\2888005364.py:50: Pandas4Warning: Series.dt.dayofyear is deprecated and will be removed in a future version. Use Series.dt.day_of_year instead.
  season = 2*np.pi*(stamp.dt.dayofyear.to_numpy()-1)/year_length
C:\Users\user\AppData\Local\Temp\ipykernel_28436\2888005364.py:50: Pandas4Warning: Series.dt.dayofyear is deprecated and will be removed in a future version. Use Series.dt.day_of_year instead.
  season = 2*np.pi*(stamp.dt.dayofyear.to_numpy()-1)/year_length


2023년: 초기 반복 5/20
2023년: 초기 반복 10/20
2023년: 초기 반복 15/20
2023년: 초기 반복 20/20
2023년: 1/50벌 저장 (반복 22)
2023년: 2/50벌 저장 (반복 24)
2023년: 초기 반복 25/20
2023년: 3/50벌 저장 (반복 26)
2023년: 4/50벌 저장 (반복 28)
2023년: 5/50벌 저장 (반복 30)
2023년: 6/50벌 저장 (반복 32)
2023년: 7/50벌 저장 (반복 34)
2023년: 초기 반복 35/20
2023년: 8/50벌 저장 (반복 36)
2023년: 9/50벌 저장 (반복 38)
2023년: 10/50벌 저장 (반복 40)
2023년: 11/50벌 저장 (반복 42)
2023년: 12/50벌 저장 (반복 44)
2023년: 초기 반복 45/20
2023년: 13/50벌 저장 (반복 46)
2023년: 14/50벌 저장 (반복 48)
2023년: 15/50벌 저장 (반복 50)
2023년: 16/50벌 저장 (반복 52)
2023년: 17/50벌 저장 (반복 54)
2023년: 초기 반복 55/20
2023년: 18/50벌 저장 (반복 56)
2023년: 19/50벌 저장 (반복 58)
2023년: 20/50벌 저장 (반복 60)
2023년: 21/50벌 저장 (반복 62)
2023년: 22/50벌 저장 (반복 64)
2023년: 초기 반복 65/20
2023년: 23/50벌 저장 (반복 66)
2023년: 24/50벌 저장 (반복 68)
2023년: 25/50벌 저장 (반복 70)
2023년: 26/50벌 저장 (반복 72)
2023년: 27/50벌 저장 (반복 74)
2023년: 초기 반복 75/20
2023년: 28/50벌 저장 (반복 76)
2023년: 29/50벌 저장 (반복 78)
2023년: 30/50벌 저장 (반복 80)
2023년: 31/50벌 저장 (반복 82)
2023년: 32/50벌 저장 (반복 84)
2023년: 초기 반복 85/20
2

C:\Users\user\AppData\Local\Temp\ipykernel_28436\2888005364.py:50: Pandas4Warning: Series.dt.dayofyear is deprecated and will be removed in a future version. Use Series.dt.day_of_year instead.
  season = 2*np.pi*(stamp.dt.dayofyear.to_numpy()-1)/year_length
C:\Users\user\AppData\Local\Temp\ipykernel_28436\2888005364.py:50: Pandas4Warning: Series.dt.dayofyear is deprecated and will be removed in a future version. Use Series.dt.day_of_year instead.
  season = 2*np.pi*(stamp.dt.dayofyear.to_numpy()-1)/year_length


2024년: 초기 반복 5/20
2024년: 초기 반복 10/20
2024년: 초기 반복 15/20
2024년: 초기 반복 20/20
2024년: 1/50벌 저장 (반복 22)
2024년: 2/50벌 저장 (반복 24)
2024년: 초기 반복 25/20
2024년: 3/50벌 저장 (반복 26)
2024년: 4/50벌 저장 (반복 28)
2024년: 5/50벌 저장 (반복 30)
2024년: 6/50벌 저장 (반복 32)
2024년: 7/50벌 저장 (반복 34)
2024년: 초기 반복 35/20
2024년: 8/50벌 저장 (반복 36)
2024년: 9/50벌 저장 (반복 38)
2024년: 10/50벌 저장 (반복 40)
2024년: 11/50벌 저장 (반복 42)
2024년: 12/50벌 저장 (반복 44)
2024년: 초기 반복 45/20
2024년: 13/50벌 저장 (반복 46)
2024년: 14/50벌 저장 (반복 48)
2024년: 15/50벌 저장 (반복 50)
2024년: 16/50벌 저장 (반복 52)
2024년: 17/50벌 저장 (반복 54)
2024년: 초기 반복 55/20
2024년: 18/50벌 저장 (반복 56)
2024년: 19/50벌 저장 (반복 58)
2024년: 20/50벌 저장 (반복 60)
2024년: 21/50벌 저장 (반복 62)
2024년: 22/50벌 저장 (반복 64)
2024년: 초기 반복 65/20
2024년: 23/50벌 저장 (반복 66)
2024년: 24/50벌 저장 (반복 68)
2024년: 25/50벌 저장 (반복 70)
2024년: 26/50벌 저장 (반복 72)
2024년: 27/50벌 저장 (반복 74)
2024년: 초기 반복 75/20
2024년: 28/50벌 저장 (반복 76)
2024년: 29/50벌 저장 (반복 78)
2024년: 30/50벌 저장 (반복 80)
2024년: 31/50벌 저장 (반복 82)
2024년: 32/50벌 저장 (반복 84)
2024년: 초기 반복 85/20
2

In [11]:
import runpy

# 전체 MICE 프로그램 불러오기
program = runpy.run_path(
    r"C:\다중회귀분석\outputs\MICE검증코드\mice_validation_runner.py"
)

# 검증자료별로 대치자료 50벌 생성
program["run_validation"](
    root=r"C:\WORK\stay_multiregression\data\MICE_검증",
    years=[2022, 2023, 2024],
    m=50,          # 검증연도별 대치자료 수
    burn_in=20,    # 저장하기 전 초기 반복 횟수
    gap=2,        # 이후 2회 갱신할 때마다 한 벌 저장
    seed=20261107,
)

FileExistsError: C:\WORK\stay_multiregression\data\MICE_검증\2022\MICE결과: 기존 CSV를 별도 보관한 뒤 실행하세요.

3. 새로 만든 대치 결과 평가
검증용 대치자료가 준비된 다음 실행.

In [12]:
from IPython.display import display

# 원래 관측값 불러오기
truth = pd.read_csv(
    validation_dir / "원래_관측자료.csv",
    encoding="utf-8-sig",
)

truth["관측시각"] = pd.to_datetime(truth["관측시각"])
truth = truth.set_index(keys).sort_index()

records = []

for year in [2022, 2023, 2024]:
    folder = validation_dir / str(year)
    files = sorted((folder / "MICE결과").glob("*.csv"))

    if len(files) != 50:
        raise ValueError(
            f"{year}년 검증용 대치파일이 50개 필요합니다. "
            f"현재 {len(files)}개입니다."
        )

    # 인위적으로 가린 위치
    mask = pd.read_csv(
        folder / "평가위치.csv",
        encoding="utf-8-sig",
    )

    mask["관측시각"] = pd.to_datetime(mask["관측시각"])
    mask = mask.set_index(keys).reindex(truth.index)

    for col in weather_cols:
        mask[col] = (
            mask[col].astype(str).str.lower().eq("true")
        )

    for number, file in enumerate(files, start=1):
        completed = pd.read_csv(
            file,
            encoding="utf-8-sig",
        )

        completed["관측시각"] = pd.to_datetime(
            completed["관측시각"]
        )
        completed["항만"] = completed["항만"].str.strip()

        if completed.duplicated(keys).any():
            raise ValueError(
                f"{file.name}: 항만·관측시각이 중복되어 있습니다."
            )

        completed = completed.set_index(keys).sort_index()

        if not completed.index.equals(truth.index):
            raise ValueError(
                f"{file.name}: 항만 또는 관측시각 범위가 다릅니다."
            )

        for col in weather_cols:
            # 가리지 않은 원래 관측값은 유지되어야 함
            unchanged = truth[col].notna() & ~mask[col]

            if col in numeric_cols:
                actual_all = pd.to_numeric(
                    truth[col], errors="raise"
                )
                filled_all = pd.to_numeric(
                    completed[col], errors="raise"
                )

                if not np.allclose(
                    actual_all[unchanged],
                    filled_all[unchanged],
                    rtol=1e-10,
                    atol=1e-10,
                ):
                    raise ValueError(
                        f"{file.name}: 원래 관측값이 변경됐습니다: {col}"
                    )
            else:
                if not (
                    truth.loc[unchanged, col].astype(str)
                    == completed.loc[unchanged, col].astype(str)
                ).all():
                    raise ValueError(
                        f"{file.name}: 원래 풍향 관측값이 변경됐습니다."
                    )

            # 항만별로 대치값과 가려둔 실제값 비교
            for port in ["북항", "신항"]:
                selected = (
                    mask[col]
                    & (
                        truth.index.get_level_values("항만")
                        == port
                    )
                )

                if not selected.any():
                    continue

                actual = truth.loc[selected, col]
                filled = completed.loc[selected, col]

                if filled.isna().any():
                    raise ValueError(
                        f"{file.name}: 평가 위치에 결측이 남았습니다: {col}"
                    )

                row = {
                    "검증연도": year,
                    "항만": port,
                    "변수": col,
                    "대치번호": number,
                    "평가개수": len(actual),
                }

                if col in numeric_cols:
                    actual = pd.to_numeric(
                        actual
                    ).to_numpy(dtype=float)

                    filled = pd.to_numeric(
                        filled
                    ).to_numpy(dtype=float)

                    if not np.isfinite(filled).all():
                        raise ValueError(
                            f"{file.name}: 대치값에 무한값이 있습니다."
                        )

                    error = filled - actual

                    row.update({
                        "MAE": np.mean(np.abs(error)),
                        "RMSE": np.sqrt(np.mean(error ** 2)),
                        "평균편향": np.mean(error),
                        "실제95분위수": np.quantile(actual, 0.95),
                        "대치95분위수": np.quantile(filled, 0.95),
                    })

                else:
                    row["풍향일치율"] = np.mean(
                        actual.astype(str).to_numpy()
                        == filled.astype(str).to_numpy()
                    )

                records.append(row)

        print(f"{year}년: {number}/50 평가 완료")

validation_detail = pd.DataFrame(records)

# 수치형 기상변수 평가표
numeric_summary = (
    validation_detail[
        validation_detail["변수"].isin(numeric_cols)
    ]
    .groupby(["검증연도", "항만", "변수"])
    .agg(
        평가개수=("평가개수", "first"),
        평균MAE=("MAE", "mean"),
        평균RMSE=("RMSE", "mean"),
        평균편향=("평균편향", "mean"),
        실제95분위수=("실제95분위수", "mean"),
        평균대치95분위수=("대치95분위수", "mean"),
    )
)

print("\n수치형 기상변수 복원 결과")
display(
    numeric_summary.style.format({
        "평가개수": "{:,.0f}",
        "평균MAE": "{:.3f}",
        "평균RMSE": "{:.3f}",
        "평균편향": "{:.3f}",
        "실제95분위수": "{:.3f}",
        "평균대치95분위수": "{:.3f}",
    })
)

# 풍향 평가표
direction_summary = (
    validation_detail[
        validation_detail["변수"] == "풍향(8방위)"
    ]
    .groupby(["검증연도", "항만"])
    .agg(
        평가개수=("평가개수", "first"),
        평균풍향일치율=("풍향일치율", "mean"),
    )
)

print("\n풍향 복원 결과")
display(
    direction_summary.style.format({
        "평가개수": "{:,.0f}",
        "평균풍향일치율": "{:.1%}",
    })
)

2022년: 1/50 평가 완료
2022년: 2/50 평가 완료
2022년: 3/50 평가 완료
2022년: 4/50 평가 완료
2022년: 5/50 평가 완료
2022년: 6/50 평가 완료
2022년: 7/50 평가 완료
2022년: 8/50 평가 완료
2022년: 9/50 평가 완료
2022년: 10/50 평가 완료
2022년: 11/50 평가 완료
2022년: 12/50 평가 완료
2022년: 13/50 평가 완료
2022년: 14/50 평가 완료
2022년: 15/50 평가 완료
2022년: 16/50 평가 완료
2022년: 17/50 평가 완료
2022년: 18/50 평가 완료
2022년: 19/50 평가 완료
2022년: 20/50 평가 완료
2022년: 21/50 평가 완료
2022년: 22/50 평가 완료
2022년: 23/50 평가 완료
2022년: 24/50 평가 완료
2022년: 25/50 평가 완료
2022년: 26/50 평가 완료
2022년: 27/50 평가 완료
2022년: 28/50 평가 완료
2022년: 29/50 평가 완료
2022년: 30/50 평가 완료
2022년: 31/50 평가 완료
2022년: 32/50 평가 완료
2022년: 33/50 평가 완료
2022년: 34/50 평가 완료
2022년: 35/50 평가 완료
2022년: 36/50 평가 완료
2022년: 37/50 평가 완료
2022년: 38/50 평가 완료
2022년: 39/50 평가 완료
2022년: 40/50 평가 완료
2022년: 41/50 평가 완료
2022년: 42/50 평가 완료
2022년: 43/50 평가 완료
2022년: 44/50 평가 완료
2022년: 45/50 평가 완료
2022년: 46/50 평가 완료
2022년: 47/50 평가 완료
2022년: 48/50 평가 완료
2022년: 49/50 평가 완료
2022년: 50/50 평가 완료
2023년: 1/50 평가 완료
2023년: 2/50 평가 완료
2023년: 3/50 평가 완료
2023년


풍향 복원 결과


,,평가개수,평균풍향일치율
검증연도,항만,,
2022,북항,26,19.9%
2023,북항,33,25.7%
2024,북항,36,25.1%


In [13]:
import runpy

# 비교 검증 프로그램 불러오기
validation_program = runpy.run_path(
    r"C:\다중회귀분석\outputs\MICE검증코드\regression_validation_compare.py"
)

# 2022·2023·2024년 검증자료에서 삭제/MICE 비교
validation_results = validation_program["run_regression_validation"](
    project=r"C:\WORK\stay_multiregression",
    years=[2022, 2023, 2024],
    m=50,
    end_side="left",             # 접안일시 <= 관측시각 < 출항일시
    deletion_rule="aggregate",  # 기상 집계 후 결측인 선박 제외
)

전체 선박 33,994건 / 원관측이 모두 있는 기준 표본 27,727건
출항시각 경계: left / 삭제 규칙: aggregate
2022: 1/50 회귀 검증 완료
2022: 2/50 회귀 검증 완료
2022: 3/50 회귀 검증 완료
2022: 4/50 회귀 검증 완료
2022: 5/50 회귀 검증 완료
2022: 6/50 회귀 검증 완료
2022: 7/50 회귀 검증 완료
2022: 8/50 회귀 검증 완료
2022: 9/50 회귀 검증 완료
2022: 10/50 회귀 검증 완료
2022: 11/50 회귀 검증 완료
2022: 12/50 회귀 검증 완료
2022: 13/50 회귀 검증 완료
2022: 14/50 회귀 검증 완료
2022: 15/50 회귀 검증 완료
2022: 16/50 회귀 검증 완료
2022: 17/50 회귀 검증 완료
2022: 18/50 회귀 검증 완료
2022: 19/50 회귀 검증 완료
2022: 20/50 회귀 검증 완료
2022: 21/50 회귀 검증 완료
2022: 22/50 회귀 검증 완료
2022: 23/50 회귀 검증 완료
2022: 24/50 회귀 검증 완료
2022: 25/50 회귀 검증 완료
2022: 26/50 회귀 검증 완료
2022: 27/50 회귀 검증 완료
2022: 28/50 회귀 검증 완료
2022: 29/50 회귀 검증 완료
2022: 30/50 회귀 검증 완료
2022: 31/50 회귀 검증 완료
2022: 32/50 회귀 검증 완료
2022: 33/50 회귀 검증 완료
2022: 34/50 회귀 검증 완료
2022: 35/50 회귀 검증 완료
2022: 36/50 회귀 검증 완료
2022: 37/50 회귀 검증 완료
2022: 38/50 회귀 검증 완료
2022: 39/50 회귀 검증 완료
2022: 40/50 회귀 검증 완료
2022: 41/50 회귀 검증 완료
2022: 42/50 회귀 검증 완료
2022: 43/50 회귀 검증 완료
2022: 44/50 회귀 검증 완료
2022: 45/50

검증연도,모형,기준 표본,삭제 표본,MICE 표본,동일 평가 표본,삭제 회귀함수 복원 RMSE(시간),MICE 회귀함수 복원 RMSE(시간),기준에 가까운 방법
2022,1,27727,26712,27727,1028,0.0425,0.0000,MICE
2022,2,27727,26712,27727,1028,0.0364,0.0000,MICE
2022,3,27727,26712,27727,1028,0.0453,0.0000,MICE
2022,4,27727,26712,27727,1028,0.1181,0.0254,MICE
2023,1,27727,26673,27727,1073,0.0328,0.0000,MICE
2023,2,27727,26673,27727,1073,0.0151,0.0000,MICE
2023,3,27727,26673,27727,1073,0.0196,0.0000,MICE
2023,4,27727,26673,27727,1073,0.0394,0.0441,삭제
2024,1,27727,26477,27727,1250,0.0559,0.0000,MICE
2024,2,27727,26477,27727,1250,0.0423,0.0000,MICE



변수별 기준 회귀계수 재현 비교


2022년: MICE가 더 가까움
2023년: 삭제가 조금 더 가까움
2024년: 거의 비슷함

In [14]:
import pandas as pd
from IPython.display import display, Markdown

# 현재 분석·검증 결과로 구성한 발표용 요약표
selection_table = pd.DataFrame({
    "비교 항목": [
        "본 분석의 표본 수",
        "2022년 회귀함수 복원 RMSE",
        "2023년 회귀함수 복원 RMSE",
        "2024년 회귀함수 복원 RMSE",
    ],
    "결측 삭제": [
        "32,014건",
        "0.1181시간",
        "0.0394시간",
        "0.0715시간",
    ],
    "MICE": [
        "33,994건",
        "0.0254시간",
        "0.0441시간",
        "0.0706시간",
    ],
    "해석": [
        "MICE에서 1,980건 추가 활용",
        "MICE가 기준 회귀식에 더 가까움",
        "삭제가 더 가까움 — 차이 0.0047시간",
        "두 방법이 거의 유사함",
    ],
})

display(Markdown("### MICE를 주 분석으로 선택한 근거"))

display(
    selection_table.style
    .hide(axis="index")
    .set_table_styles([
        {
            "selector": "table",
            "props": [
                ("border-collapse", "collapse"),
                ("font-family", "Malgun Gothic"),
                ("font-size", "17px"),
            ],
        },
        {
            "selector": "th",
            "props": [
                ("background-color", "#203864"),
                ("color", "white"),
                ("padding", "14px 18px"),
                ("text-align", "center"),
            ],
        },
        {
            "selector": "td",
            "props": [
                ("padding", "14px 18px"),
                ("border-bottom", "1px solid #d9e2f3"),
                ("text-align", "center"),
                ("color", "#222222"),
            ],
        },
        {
            "selector": "tbody tr:nth-child(even)",
            "props": [
                ("background-color", "#f2f5fa"),
            ],
        },
    ])
    .set_properties(
        subset=["MICE"],
        **{"font-weight": "bold"}
    )
    .set_properties(
        subset=["비교 항목", "해석"],
        **{"text-align": "left"}
    )
)

display(Markdown(
    "**선택 이유:** 결측에 따른 표본 손실을 줄이고, "
    "가리기 전 회귀관계를 대체로 재현했으며 "
    "2022년 검증에서는 삭제법보다 복원 오차가 작아 MICE를 주 분석으로 선택하였다. "
    "결측 삭제 결과는 민감도 분석으로 함께 제시한다."
))

display(Markdown(
    "*주: 검증 RMSE는 실제 체류시간에 대한 예측 오차가 아니라 "
    "가리기 전 기준 회귀함수와의 차이이며, 작을수록 기준에 가깝다. "
    "기상변수를 포함한 모형 4만 비교하였다. "
    "세 연도 모두에서 MICE가 우수했던 것은 아니다.*"
))

### MICE를 주 분석으로 선택한 근거

비교 항목,결측 삭제,MICE,해석
본 분석의 표본 수,"32,014건","33,994건","MICE에서 1,980건 추가 활용"
2022년 회귀함수 복원 RMSE,0.1181시간,0.0254시간,MICE가 기준 회귀식에 더 가까움
2023년 회귀함수 복원 RMSE,0.0394시간,0.0441시간,삭제가 더 가까움 — 차이 0.0047시간
2024년 회귀함수 복원 RMSE,0.0715시간,0.0706시간,두 방법이 거의 유사함


**선택 이유:** 결측에 따른 표본 손실을 줄이고, 가리기 전 회귀관계를 대체로 재현했으며 2022년 검증에서는 삭제법보다 복원 오차가 작아 MICE를 주 분석으로 선택하였다. 결측 삭제 결과는 민감도 분석으로 함께 제시한다.

*주: 검증 RMSE는 실제 체류시간에 대한 예측 오차가 아니라 가리기 전 기준 회귀함수와의 차이이며, 작을수록 기준에 가깝다. 기상변수를 포함한 모형 4만 비교하였다. 세 연도 모두에서 MICE가 우수했던 것은 아니다.*